#### Graph Construction
Think of this module as constructing a house.

* state = building materials
* Nodes = Rooms
* Edges = Hall ways connecting to rooms
* StateGraph = the complete blue print
* Compile = Final inspection
* Invoke = Someone entering and using the house.


#### What is Graph?
* Graph is a simply collection of 
    * Nodes
    * Edges
    * State
working together.

Imagine a simple chatbot:
``` markdown
User Question
      │
      ▼
Understand Question
      │
      ▼
Search Knowledge
      │
      ▼
Generate Answer
      │
      ▼
Return Response
```
Everyhting together is called Graph



#### State Graph
A state graph is the main object used to build workflows in langgraph.

it acts as a container for 
* State
* Nodes
* Edges
* Start Point
* End Point

Think of it like a project manager.

##### Visualization

``` markdown
             StateGraph

        +----------------------+

         Nodes

         Edges

         State

         START

         END

        +----------------------+
```
Everything lives inside the Graph.

Creating a StateGraph

In [4]:
from typing import TypedDict

# defined the state schema
class GraphState(TypedDict):
    name:str

# define the blue print that create the Graph
from langgraph.graph import StateGraph

graph = StateGraph(GraphState)
print(graph)
# it means "Create a workflow whose shared memory follows the graph state structure"

In [5]:
## Example 

from typing import TypedDict

from langgraph.graph import StateGraph

# defining the state is to know the graph what information inside it.
class StudentState(TypedDict):
    name:str
    marks:int
    grade:str

# now every node knows these values exists


#### START Node

Every node needs a starting point like airport,railwaystation,movie everything has a beggining right. in langgraph the beginning is called ``START``. the graph always starts from here.

``` markdown
START
  │
  ▼
Read Input
```

``` python
from langgraph.graph import START
```
later we connect it to edge

``` python
graph.add_edge(START,"read_input")
```


#### END Node
every graph needs an ending. 

read input --> generate response --> END.
Once end is reached the excecution stops.

``` python
from langgraph.graph import END

# connect to an edge
graph.add_edge("generate_response",END)
```

#### Complete flow
``` markdown         
          START

             │

             ▼

       Read Input

             │

             ▼

      Process Data

             │

             ▼

      Generate Output

             │

             ▼

            END
```

#### Adding Nodes.
Now we need a workers. Suppose we have, How does these nodes are getting existed for this basically we are registering them. using ``add_node``

In [ ]:
def greet(state):
    print("Hello")
    return state

# another node
def bye(state):
    print("Goodbye")
    return state

# syntax # for single node
graph.add_node("greet", greet)

## for multiple nodes
graph.add_node("greet",greet)
graph.add_node("bye", bye)

#### Adding Edges 
Edges connect Nodes.

```Start --> greet --> bye-->end```

``` python
graph.add_edge(START,"greet")
graph.add_edge("greet","bye")
graph.add_edge("bye", END)
```

Now everything is connected.

In [11]:
## Example:

# create State
from typing import TypedDict

class GraphState(TypedDict):
    name: str

In [12]:
# step 2: Create Graph
from langgraph.graph import StateGraph

graph = StateGraph(GraphState)

In [15]:
# step 3 create nodes 
def greet(state):
    print("Hello", state["name"])
    return state

def goodbye(state):
    print("Goodbye", state["name"])
    return state

In [ ]:
# step 4: add nodes 

graph.add_node("greet", greet)
graph.add_node("goodbye", goodbye)

In [ ]:
# step 5: connect nodes
from langgraph.graph import START, END

graph.add_edge(START, "greet")
graph.add_edge("greet", "goodbye")
graph.add_edge("goodbye", END)

#### Compiling Graph
To excecutes the Graph we need the compiler.

``` python
compile()

# code
app = graph.complile()
# now the graph becomes excecutable
```

#### Behind the Scenes
when compile runs, Langgraph checks
* Are all nodes connected?
* Does START Exist?
* Does END Exist?
* Are there invalid edges?
* Are there Missing nodes?

if everything is correct, it creates an excecutable graph.

In [17]:
### Complete code Example

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# -----------------------------------
# Step 1: Define the shared state
# -----------------------------------
class GraphState(TypedDict):
    name: str

# -----------------------------------
# Step 2: Define node functions
# -----------------------------------
def greet(state: GraphState):
    print(f"Hello {state['name']}")
    return state


def goodbye(state: GraphState):
    print(f"Goodbye {state['name']}")
    return state


# -----------------------------------
# Step 3: Create the graph
# -----------------------------------
builder = StateGraph(GraphState)


# -----------------------------------
# Step 4: Register nodes
# -----------------------------------
builder.add_node("greet", greet)
builder.add_node("goodbye", goodbye)


# -----------------------------------
# Step 5: Connect the nodes
# -----------------------------------
builder.add_edge(START, "greet")
builder.add_edge("greet", "goodbye")
builder.add_edge("goodbye", END)


# -----------------------------------
# Step 6: Compile the graph
# -----------------------------------
app = builder.compile()
print("===compilation===",app)

# -----------------------------------
# Step 7: Execute the graph
# -----------------------------------
result = app.invoke({"name": "Subbu"})

print("\nFinal State:")
print(result)

===compilation=== <langgraph.graph.state.CompiledStateGraph object at 0x0000028EBE30D9F0>
Hello Subbu
Goodbye Subbu

Final State:
{'name': 'Subbu'}


In [18]:
# Updating State

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class StudentState(TypedDict):
    name: str
    marks: int
    grade: str


def calculate_grade(state: StudentState):
    if state["marks"] >= 90:
        state["grade"] = "A+"
    elif state["marks"] >= 75:
        state["grade"] = "A"
    else:
        state["grade"] = "B"
    return state


def display_result(state: StudentState):
    print(
        f"{state['name']} scored {state['marks']} and received grade {state['grade']}"
    )
    return state


builder = StateGraph(StudentState)

builder.add_node("calculate_grade", calculate_grade)
builder.add_node("display_result", display_result)

builder.add_edge(START, "calculate_grade")
builder.add_edge("calculate_grade", "display_result")
builder.add_edge("display_result", END)

app = builder.compile()

result = app.invoke(
    {
        "name": "Subbu",
        "marks": 92,
        "grade": ""
    }
)

print(result)

Subbu scored 92 and received grade A+
{'name': 'Subbu', 'marks': 92, 'grade': 'A+'}


##### Builder vs Compiled Graph

| Builder (`StateGraph`)               | Compiled Graph (`compile()`)                |
| ------------------------------------ | ------------------------------------------- |
| Used while constructing the workflow | Used to execute the workflow                |
| Add nodes                            | Cannot add new nodes                        |
| Add edges                            | Structure is fixed                          |
| Define the graph                     | Run the graph with `invoke()` or `stream()` |
| Think of it as a blueprint           | Think of it as the finished building        |


#### StateGraph
StateGraph is the container object that holds your entire graph. You pass it a state schema — a TypedDict that defines every field the graph will use. All nodes read from and write to this single shared state object.

![alt text](image-6.png)

StateGraph is just a builder — it holds your schema and the node/edge registry. Nothing executes until you call compile().

``` python
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

# Step 1 — define your state schema
class OrderState(TypedDict):
    order_id: str
    items:    list[str]
    total:    float
    status:   str
    notes:    Annotated[list[str], add]  # reducer: append on every write

# Step 2 — create the StateGraph, pass in the schema
builder = StateGraph(OrderState)

# builder is now an empty graph that knows the shape of its state
# Every node must return keys that are a subset of OrderState fields
```



#### START Node 
The START Node is virtual node that langgraph automatically creates. It represents the entry point of graph - the first node we recieves the initial input you pass to invoke(). You never define START yourself. you just wire an edge from it.

![alt text](image-7.png)

The initial state you pass to invoke() becomes the starting value of the graph's state. START just routes it to your first node — it runs no code of its own.


Method 1 Shorthand

``` python
builder.set_entry_point("validate_node")
# identical to add_edge(START, "validate_node")
```

Method 2 — explicit
``` python
from langgraph.graph import START
builder.add_edge(START, "validate_node")
# more explicit, same result
```


#### END Node
The END node is another virtual node that signals graph termination. When execution reaches any edge that points to END, the graph stops and returns the current state to the caller. You can have multiple paths converging on END.

END can also appear directly inside a conditional edge mapping as a value — execution terminates the moment any edge pointing to END is traversed.

![alt text](image-8.png)

``` python
from langgraph.graph import END

# Simple: direct edges to END
builder.add_edge("confirm_node", END)

# Conditional: multiple paths all terminate at END
def route(state):
    if state["status"] == "confirmed": return "done"
    if state["retries"] >= 3:          return "fallback"
    return "error"

builder.add_conditional_edges(
    "validate_node", route,
    {
        "done":     "confirm_node",   # → more nodes
        "fallback": "fallback_node",  # → more nodes
        "error":    END,               # → terminates immediately
    }
)
builder.add_edge("confirm_node",  END)
builder.add_edge("fallback_node", END)

```

#### Adding Nodes
add_node(name, fn) registers a callable under a name. The name is how you reference the node in edges. The callable can be a plain function, a closure, or a class instance with __call__.

![alt text](image-9.png)

``` python
# ── Form 1: plain function ──
def validate_node(state: OrderState) -> dict:
    valid = len(state["items"]) > 0
    return {"status": "valid" if valid else "empty", "notes": ["Validated"]}

builder.add_node("validate", validate_node)

# ── Form 2: closure (factory that returns a configured function) ──
def make_pricer(tax_rate: float):
    def price_node(state: OrderState) -> dict:
        subtotal = len(state["items"]) * 9.99
        total    = round(subtotal * (1 + tax_rate), 2)
        return {"total": total, "notes": [f"Total with {tax_rate*100:.0f}% tax: ${total}"]}
    return price_node

builder.add_node("price", make_pricer(tax_rate=0.18))   # 18% GST

# ── Form 3: class with __call__ ──
class ConfirmNode:
    def __init__(self, warehouse: str):
        self.warehouse = warehouse

    def __call__(self, state: OrderState) -> dict:
        msg = f"Order {state['order_id']} routed to {self.warehouse}"
        return {"status": "confirmed", "notes": [msg]}

builder.add_node("confirm", ConfirmNode(warehouse="Bengaluru-WH-1"))

# Every node name must be unique in the graph
# The name is what you reference in add_edge() and add_conditional_edges()
```
The node name (first arg) is just a string label — it has nothing to do with the function name. But keeping them consistent makes code readable.

#### Adding Edges
Edges define the execution order. LangGraph supports three edge types: simple (always go to B after A), conditional (router function decides), and fan-out (one source, multiple simultaneous targets).

![alt text](image-10.png)

``` python
builder.add_edge(
"validate", "price"
)
# unconditional
builder.add_conditional_edges(
"validate", route_fn, {...}
)
# router picks next node

builder.add_edge("in","chunk")
builder.add_edge("in","meta")
builder.add_edge("in","detect")
# all three fire at once
```

You can mix all three edge types in a single graph. Add simple edges for the backbone, conditional edges at decision points, and fan-out edges for work you want done in parallel.

#### Compiling
builder.compile() validates the graph structure, locks it, and returns a CompiledStateGraph — the object you actually run. The builder is discarded after this step. Compile also lets you attach checkpointers and interrupt points.

![alt text](image-11.png)

``` python
# Basic compile — no options
graph = builder.compile()

# Compile with a checkpointer (enables memory between runs)
from langgraph.checkpoint.memory import MemorySaver

graph = builder.compile(
    checkpointer=MemorySaver(),        # persists state between .invoke() calls
    interrupt_before=["confirm"],       # pause before 'confirm' for human review
    interrupt_after=["validate"],       # pause after 'validate' to inspect state
)

# Visualize the compiled graph structure
print(graph.get_graph().draw_mermaid())   # prints mermaid diagram source
graph.get_graph().print_ascii()          # ASCII art in terminal
```

After compile, the graph is immutable. You cannot add more nodes or edges. To change the structure you must go back to the builder and compile again.



#### Running
A compiled graph has three main run modes: invoke() runs to completion and returns the final state; stream() yields intermediate results after each node; batch() runs multiple inputs in parallel.

![alt text](image-12.png)

``` python
# ── invoke() — run to completion, returns final state ──
result = graph.invoke({
    "order_id": "ORD-001",
    "items":    ["book", "pen"],
    "total":    0.0,
    "status":   "",
    "notes":    []
})
# result = {"order_id": "ORD-001", "status": "confirmed", "total": 23.58, ...}

# ── stream() — yields after each node ──
for chunk in graph.stream(initial_state):
    print(chunk)
# {"validate": {"status": "valid", "notes": [...]}} 
# {"price":    {"total": 23.58,   "notes": [...]}}
# {"confirm":  {"status": "confirmed", "notes": [...]}}

# ── stream() with stream_mode="values" — full state snapshot after each node ──
for state in graph.stream(initial_state, stream_mode="values"):
    print(f"status={state['status']}  total={state['total']}")

# ── config dict — pass runtime metadata ──
result = graph.invoke(
    initial_state,
    config={"configurable": {"thread_id": "session-42"}}
)

# ── batch() — multiple inputs, concurrent ──
orders = [order1_state, order2_state, order3_state]
results = graph.batch(orders)   # returns [result1, result2, result3]
```

stream_mode="updates" (default) gives you only the dict each node returned. stream_mode="values" gives you the full merged state after each node — useful when you want to watch state evolve.